<a href="https://colab.research.google.com/github/pcmay/ALyzer3D.AI/blob/main/ALyzer3D_AI_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




<div style="display: flex; justify-content: space-between; align-items: center;">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ALyzer3D.AI_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ColabFold_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
</div>



Welcome to **ALyzer3D.AI BATCH**. This notebook allows you to predict the amyloidogenicity of the VL domains of a list of light chains. The tool will first generate 3D structures with [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb) and then automatically analyze them with the ALyzer3D.AI model.

**Instructions:**

1. **Enter Your Sequences**: In the first cell (sequences_input), paste the amino acid sequences of your light chains' VL domains. Format: >ID1:sequence1;>ID2:sequence2;>ID3:sequence3;...
2. **Select a GPU**: Click Runtime, select Change runtime type, select T4 GPU (or any GPU option available). Click Save.
3. **Run Everything**: Click on the menu Runtime -> Run all.

The notebook will now execute all the steps for you: it will install dependencies, run the ColabFold structure predictions and perform the ALyzer3D.AI analyses on the resulting top-ranked structures. At Step 3, you will be able to download a CSV file with the results.


---



In [ ]:
#@title Install Dependencies and Mount Google Drive
from google.colab import drive
import os
import sys
from sys import version_info


# Check if ColabFold and its dependencies are already installed
if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    # Install ColabFold
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")

    # Fix for TensorFlow "undefined symbol" error
    os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")

    # Create symbolic links
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
    os.system("touch COLABFOLD_READY")

# Install ALyzer3D.AI dependencies
print("Installing ALyzer3D.AI and its dependencies...")
os.system("git clone https://github.com/petercmay89/ALyzer3D.AI.git > /dev/null 2>&1")
sys.path.insert(0, '/content/ALyzer3D.AI')
os.system("pip install -q transformers scikit-learn joblib > /dev/null 2>&1")

In [ ]:
#@title Run Batch Prediction and Analysis

#@markdown ### Paste your sequences below (e.g., >ID1:SEQUENCE1;ID2:SEQUENCE2)
sequences_input = '>ID1:DIQMTQSPSSLSASVGDSVTITCRASHDINTYLGWLQQTPGKAPKSLIYAASTLQSGVPSRFSGGGSGTHFTLNISSLQPEDFATFYCQQYRSYPVTFGQGTRLDIK;ID2:EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKVSYLNWGDYWGQGTLVTVSS' #@param {type:"string"}
output_dir = 'colabfold_output'

# --- Standard Parameters ---
num_relax = 0
template_mode = "none"
msa_mode = "mmseqs2_uniref_env"
model_type = "auto"
pair_mode = "unpaired_paired"
num_recycles = "auto"
# ---------------------------

import os
import re
from pathlib import Path
import warnings
from IPython.display import display, HTML

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

from colabfold.download import download_alphafold_params
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from prediction_tool import AmyloidPredictor

# --- Main execution ---
# This list will store the results for Cell 3
all_results = []

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load the AI model
predictor = None
try:
    predictor = AmyloidPredictor(
        model_dir="/content/ALyzer3D.AI/champion_v8_ensemble_modelv3",
        scaler_path="/content/ALyzer3D.AI/champion_v8_ensemble_modelv3/scalar_scaler_v8_ensemble.joblib"
    )
    print("✔️ Model loaded successfully.")
except Exception as e:
    print(f"❗️ Error loading model: {e}")

# Proceed only if the model was loaded successfully
if predictor:
    # Parse the input sequences
    sequences_to_process = []
    processed_input = sequences_input.strip()
    if processed_input.startswith('>'):
        processed_input = processed_input[1:]
    entries = processed_input.split(';')

    for entry in entries:
        if ':' in entry:
            parts = entry.split(':', 1)
            if len(parts) == 2 and parts[0].strip() and parts[1].strip():
                sequences_to_process.append({'id': parts[0].strip(), 'sequence': parts[1].strip()})
            else:
                print(f"⚠️ Skipping malformed entry part: '{entry}'")
        elif entry.strip():
            print(f"⚠️ Skipping malformed entry (missing ':'): '{entry}'")

    # Process each parsed sequence
    for item in sequences_to_process:
        query_id = item['id']
        query_sequence = "".join(item['sequence'].split())

        sanitized_jobname = re.sub(r'\W+', '', query_id)
        jobname = f"{output_dir}/{sanitized_jobname}"
        os.makedirs(jobname, exist_ok=True)

        queries_path = os.path.join(jobname, f"{sanitized_jobname}.csv")
        with open(queries_path, "w") as text_file:
            text_file.write(f"id,sequence\n{sanitized_jobname},{query_sequence}")

        print(f"\n{'='*50}\nProcessing ID: {query_id}")
        print(f"Sequence length: {len(query_sequence)}")

        result_dir = Path(jobname)
        setup_logging(result_dir.joinpath("log.txt"))

        queries, is_complex = get_queries(queries_path)
        model_type_run = set_model_type(is_complex, model_type)
        num_recycles_parsed = None if num_recycles == "auto" else int(num_recycles)

        download_alphafold_params(model_type_run, Path("."))

        run(
            queries=queries, result_dir=result_dir, use_templates=(template_mode != "none"),
            num_relax=num_relax, msa_mode=msa_mode, model_type=model_type_run,
            num_models=1, num_recycles=num_recycles_parsed, model_order=[1],
            is_complex=is_complex, data_dir=Path("."), keep_existing_results=False,
            rank_by="auto", pair_mode=pair_mode, stop_at_score=100.0,
            zip_results=False, user_agent="colabfold/google-colab-main",
        )

        pdb_file = next(Path(jobname).glob("*_unrelaxed_rank_001*.pdb"), None)
        json_file = next(Path(jobname).glob("*_scores_rank_001*.json"), None)

        if pdb_file and json_file:
            print(f"Analyzing results for {query_id}...")
            result = predictor.predict(pdb_path=str(pdb_file), json_path=str(json_file))

            if result.get("error"):
                print(f"❗️ An error occurred during analysis: {result['error']}")
                all_results.append({
                    "ID": query_id, "Sequence": query_sequence,
                    "Prediction": f"Analysis Error: {result['error']}",
                    "Confidence Score (%)": "N/A"
                })
            else:
                prob = result['prediction_probability']
                confidence_percent = prob * 100
                print(f"\n--- Output for {query_id} ---")
                print(f"  Prediction: {result['prediction_label']}")
                print(f"  Confidence Score: {confidence_percent:.2f}%")
                print("--------------------------\n")
                all_results.append({
                    "ID": query_id,
                    "Sequence": result['sequence'],
                    "Prediction": result['prediction_label'],
                    "Confidence Score (%)": f"{confidence_percent:.2f}"
                })
        else:
            print(f"❗️ Error: Could not find ColabFold output files for {sanitized_jobname}.")
            all_results.append({
                "ID": query_id, "Sequence": query_sequence,
                "Prediction": "Processing Error",
                "Confidence Score (%)": "N/A"
            })

    print("\n\n✅ Batch processing complete.")

In [ ]:
#@title Download Results as CSV
import pandas as pd
from google.colab import files

# Check if the 'all_results' list exists and has content
if 'all_results' in locals() and all_results:
    # Convert the list of dictionaries to a pandas DataFrame
    results_df = pd.DataFrame(all_results)

    # Define the CSV filename
    csv_filename = 'alyzer3d_analysis_results.csv'

    # Save the DataFrame to a CSV file
    results_df.to_csv(csv_filename, index=False)

    print(f"✅ Results have been saved to '{csv_filename}'.")
    print("Starting download...")

    # Trigger the file download in the browser
    files.download(csv_filename)
else:
    print("❗️ No results found to download. Please run the 'Batch Prediction and Analysis' cell (Cell 2) first.")